# Объем внедомашнего инвентаря
Пример расчета объема внедомашнего инвентаря по телекомпаниям за месяц 

## Описание задачи и условий расчета
- Big TV Россия 0+
- Период: 01.07.2025-31.07.2025
- ЦА: Все 4+
- Каналы: все телекомпании, кроме рекламных и интернет
- Ролики: Блок содержание - коммерческий; Блок распространение - орбитальный, сетевой
- Статистики: SpotByBreaksStandSalesRtgPerSum

## Инициализация

При построении отчета первый шаг в любом ноутбуке - загрузка библиотек, которые помогут обращаться к TVI API и работать с данными.

Выполните следующую ячейку, для этого перейдите в нее и нажмите Ctrl+Enter

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import re
import json
import datetime
import time
import pandas as pd
#import matplotlib.pyplot as plt
from IPython.display import JSON
from mediascope_api.core import utils

from mediascope_api.core import net as mscore
from mediascope_api.mediavortex import tasks as cwt
from mediascope_api.mediavortex import catalogs as cwc

# Настраиваем отображение

# Включаем отображение всех колонок
pd.set_option('display.max_columns', None)
# Задаем максимальное количество выводимых строк. Раскомментируйте нужную строку
# 200 строк
# pd.set_option("display.max_rows", 200)
# Отображаем все строки. ВАЖНО! Отображение большого DataFrame требует много ресурсов
# pd.set_option("display.max_rows", None)

# Cоздаем объекты для работы с TVI API
mnet = mscore.MediascopeApiNetwork()
mtask = cwt.MediaVortexTask()
cats = cwc.MediaVortexCats()

## Справочники

Получим идентификаторы, которые будут использоваться для формирования условий расчета

In [ ]:
# Справочник мест просмотра
cats.get_tv_location()

In [ ]:
# Блок содержание - коммерческий
cats.get_tv_breaks_content(name=['Коммерческий'])

In [ ]:
# Блок распространение - орбитальный, сетевой
cats.get_tv_breaks_distribution(name=['Орбитальный','Сетевой'])

## Формирование задания
Зададим условия расчета

In [ ]:
# Задаем период
# Период указывается в виде списка ('Начало', 'Конец') в формате 'YYYY-MM-DD'. Можно указать несколько периодов
date_filter = [('2025-07-01', '2025-07-31')]

# Задаем дни недели
weekday_filter = None

# Задаем тип дня
daytype_filter = None

# Задаем ЦА
basedemo_filter = None

# Доп фильтр ЦА, нужен только в случае расчета отношения между ЦА, например, при расчете Affinity Index
targetdemo_filter = None

# Задаем место просмотра
location_filter=None

# Задаем каналы
company_filter = 'tvCompanyId IN (975,1335,1858,1859,1860,1861,1862,1865,1867,1869,1870,1871,1872,1873,1874,1875,1877,2236,2716,2742,4536,5297,5673,6638,6859,7161,12169,13835,14083)'

# Указываем фильтр программ
program_filter = None

# Фильтр блоков: Блок содержание - коммерческий; Блок распространение - орбитальный, сетевой
break_filter = 'breaksContentType = C and breaksDistributionType IN (O,N)'

# Фильтр роликов
ad_filter = None

# Задаем платформу
platform_filter = None

# Задаем playback
playbacktype_filter = None

# Указываем список срезов (группировок)
slices = [
    'tvCompanyName', # Телекомпания. Обязательный атрибут!
]

# Указываем список статистик для расчета
statistics = ['SpotByBreaksStandSalesRtgPerSum']

# Задаем условия сортировки: Телекомпания (от а до я)
sortings = {'tvCompanyName':'ASC'}

# Задаем опции расчета
options = {
    "kitId": 7, # Big TV Россия 0+
    "bigTv": True,
    "issueType": "AD", #Тип события - Ролики
}

Сформируем словарь с местами просмотра

In [ ]:
locations = {
    'a. Дом, Дача': 'locationId IN (1,2)',
    'b. Вне Дома': 'locationId IN (4)',
    'c. Дом, Дача, Вне Дома': 'locationId IN (1,2,4)',
}

Создадим комбинации

In [ ]:
combinations = utils.combine_dicts(locations)

## Расчет задания

In [ ]:
# Посчитаем задания в цикле
tasks = []
print("Отправляем задания на расчет")

# Для каждой комбинации формируем задание и отправляем на расчет
for k, v in combinations.items():
    
    # Подставляем значения в параметры
    project_name = k
    
    location_filter = v[0] # место просмотра
      
    # Формируем задание для API TV Index в формате JSON
    task_json = mtask.build_crosstab_task(date_filter=date_filter, weekday_filter=weekday_filter, 
                                          daytype_filter=daytype_filter, company_filter=company_filter, 
                                          location_filter=location_filter, basedemo_filter=basedemo_filter, 
                                          targetdemo_filter=targetdemo_filter,program_filter=program_filter, 
                                          break_filter=break_filter, ad_filter=ad_filter,
                                          platform_filter=platform_filter, playbacktype_filter=playbacktype_filter,
                                          slices=slices, statistics=statistics, sortings=sortings, options=options)

    # Для каждого этапа цикла формируем словарь с параметрами и отправленным заданием на расчет
    tsk = {}
    tsk['project_name'] = project_name    
    tsk['task'] = mtask.send_crosstab_task(task_json)
    tasks.append(tsk)
    time.sleep(2)
    print('.', end = '')
    
print(f"\nid: {[i['task']['taskId'] for i in tasks]}") 

print('')
# Ждем выполнения
print('Ждем выполнения')
tsks = mtask.wait_task(tasks)
print('Расчет завершен, получаем результат')

# Получаем результат
results = []
print('Собираем таблицу')
for t in tasks:
    tsk = t['task'] 
    df_result = mtask.result2table(mtask.get_result(tsk), project_name = t['project_name'])        
    results.append(df_result)
    print('.', end = '')
df = pd.concat(results)

# Приводим порядок столбцов в соответствие с условиями расчета
df = df[['prj_name']+slices+statistics]

df

## Настройка внешнего вида таблицы
Пропустите этот шаг, если хотите экспортировать таблицу в ее текущем виде

In [ ]:
# Формируем сводную таблицу: строки - телекомпания; столбцы - место просмотра; значения - статистики
df = pd.pivot_table(df, values=statistics,
                        index=['tvCompanyName'], 
                        columns=['prj_name'])
df

In [ ]:
# Опционально: перенести статистики в нижний уровень столбцов
df = df.reorder_levels([1, 0], axis=1).sort_index(axis=1)

df

## Сохраняем в Excel
По умолчанию файл сохраняется в папку `excel`

In [ ]:
df_info = mtask.task_builder.get_report_info()

with pd.ExcelWriter(mtask.task_builder.get_excel_filename('vnedom')) as writer:
    df.to_excel(writer, sheet_name='Report', index=True)
    df_info.to_excel(writer, sheet_name='Info', index=False)